# SentinelFlow: Exploratory Data Analysis & Feature Justification

This notebook audits and visually demonstrates the discriminative power of the features extracted by `sentinelflow.features` from raw PCAP captures.

### Objectives
1. **Ingest PCAPs**: Parse synthetic captures (`data/pcaps/*.pcap`) using `sentinelflow.parser`.
2. **Extract Features**: Compute 5-second window aggregations (`extract_window_features`) and label them against ground-truth CSVs (`label_windows`).
3. **Evaluate Separability**: Plot benign vs attack distributions for each feature.
4. **Feature Attribution & Justification**: Document the cybersecurity domain rationale behind every metric.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sentinelflow.parser import parse_pcap
from sentinelflow.features import extract_window_features, extract_bidirectional_flows, label_windows

# Locate data directory relative to repository root
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pcaps_dir = repo_root / 'data' / 'pcaps'
print(f'PCAP directory: {pcaps_dir.resolve()}')

## 1. Load PCAPs & Extract Labeled Windows

We extract features from each scenario and concatenate them into a unified evaluation dataset.

In [ ]:
scenarios = [
    ('s1_benign.pcap', 's1_benign_labels.csv'),
    ('s2_port_scan.pcap', 's2_port_scan_labels.csv'),
    ('s2_slow_scan.pcap', 's2_slow_scan_labels.csv'),
    ('s3_syn_flood.pcap', 's3_syn_flood_labels.csv'),
    ('s3_udp_flood.pcap', 's3_udp_flood_labels.csv'),
    ('s4_ssh_brute_force.pcap', 's4_ssh_brute_force_labels.csv'),
    ('s4_arp_spoof.pcap', 's4_arp_spoof_labels.csv'),
    ('s5_distributed_slow_scan.pcap', 's5_distributed_slow_scan_labels.csv'),
]

all_windows = []
all_flows = []

for pcap_name, label_name in scenarios:
    pcap_path = pcaps_dir / pcap_name
    label_path = pcaps_dir / label_name
    if not pcap_path.exists():
        continue
    
    print(f'Processing {pcap_name}...')
    pkts_df = parse_pcap(pcap_path)
    win_df = extract_window_features(pkts_df, window_size=5.0)
    
    lbl_df = pd.read_csv(label_path) if label_path.exists() else pd.DataFrame()
    labeled_win = label_windows(win_df, lbl_df)
    labeled_win['scenario'] = pcap_name.replace('.pcap', '')
    all_windows.append(labeled_win)
    
    flows = extract_bidirectional_flows(pkts_df)
    flows['scenario'] = pcap_name.replace('.pcap', '')
    all_flows.append(flows)

eda_df = pd.concat(all_windows, ignore_index=True)
flows_df = pd.concat(all_flows, ignore_index=True)
print(f'Total extracted windows: {len(eda_df):,}')
print(f'Total extracted flows: {len(flows_df):,}')
eda_df['attack_type'].value_counts()

## 2. Feature Justification & Distribution Analysis

Below we examine each extracted feature group, explaining **why** it was designed and visualizing its discriminative power across benign vs malicious traffic.

### Feature Group A: Destination Port Diversity (`unique_dst_ports`)

- **Cybersecurity Rationale**: In standard enterprise operations, clients communicate with a narrow set of well-known services (e.g. 80, 443, 53). Port scanning (MITRE `T1046`) systematically sweeps hundreds of ports in rapid succession.
- **Expected Behavior**: Benign traffic rarely contacts $>3$ unique destination ports per 5-second window, whereas a port scan surges to 50–200 unique ports per window.

In [ ]:
# Plot unique_dst_ports across attack classes
fig = px.box(
    eda_df,
    x='attack_type',
    y='unique_dst_ports',
    color='is_malicious',
    title='Distribution of Unique Destination Ports per 5s Window',
    labels={'unique_dst_ports': 'Unique Dst Ports', 'attack_type': 'Traffic Classification'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group B: Volume & Transmission Rates (`pps` & `bps`)

- **Cybersecurity Rationale**: Direct Network Denial of Service (MITRE `T1498.001`) attacks (such as SYN floods and UDP floods) attempt to exhaust socket buffers and network bandwidth.
- **Expected Behavior**: Benign windows exhibit $<10$ packets/sec, whereas active DoS windows spike to $>100$ packets/sec and orders of magnitude higher bits/sec.

In [ ]:
# Compare PPS (Packets Per Second) across scenarios
fig = px.box(
    eda_df,
    x='attack_type',
    y='pps',
    color='is_malicious',
    title='Packets Per Second (pps) Across Traffic Classes',
    labels={'pps': 'Packets / Second', 'attack_type': 'Traffic Class'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group C: TCP Handshake Dynamics (`syn_count`, `synack_ratio`, `syn_without_ack_fraction`)

- **Cybersecurity Rationale**: Legitimate TCP communication adheres to RFC 793, completing the 3-way handshake (`SYN` $\rightarrow$ `SYN-ACK` $\rightarrow$ `ACK`). In contrast:
  1. **SYN Floods**: Transmit thousands of SYNs while ignoring SYN-ACKs, driving `syn_without_ack_fraction` to $\approx 1.0$.
  2. **SYN Stealth Scans**: Abort half-open sockets with RST rather than ACK.
- **Expected Behavior**: Benign traffic maintains balanced `synack_ratio` (near 1.0) and near-zero unacknowledged SYNs.

In [ ]:
fig = px.scatter(
    eda_df,
    x='syn_count',
    y='syn_without_ack_fraction',
    color='attack_type',
    title='SYN Volume vs Unacknowledged SYN Fraction',
    labels={'syn_count': 'SYN Count in Window', 'syn_without_ack_fraction': 'Unacknowledged SYN Fraction'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group D: Session Ephemerality (`short_connection_fraction`)

- **Cybersecurity Rationale**: Measures the fraction of connections that terminate after $\le 3$ packets or with a TCP RST.
- **Relevance**: Automated dictionary attacks (e.g. SSH Brute Force, MITRE `T1110.001`) and port scans establish rapid, failed connections that teardown immediately, resulting in `short_connection_fraction` $\approx 1.0$.

In [ ]:
fig = px.box(
    eda_df[eda_df['tcp_share'] > 0],
    x='attack_type',
    y='short_connection_fraction',
    color='is_malicious',
    title='Short Connection Fraction by Traffic Type',
    labels={'short_connection_fraction': 'Fraction of Short/Reset Connections'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group E: Service Port Concentration (`connections_to_same_dst_port_max`)

- **Cybersecurity Rationale**: Unlike port scans that distribute across many ports, brute force and application floods concentrate dozens of connection attempts against a single target service (e.g., SSH port 22 or HTTP port 80).
- **Expected Behavior**: Benign endpoints establish 1–3 connections per service per window; SSH brute force generates $>10$ attempts per window to port 22.

In [ ]:
fig = px.box(
    eda_df,
    x='attack_type',
    y='connections_to_same_dst_port_max',
    color='is_malicious',
    title='Maximum Connections to a Single Destination Port in Window',
    labels={'connections_to_same_dst_port_max': 'Max Conns to Single Port'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group F: Layer-2 Poisoning (`arp_reply_count` & `mac_binding_changes`)

- **Cybersecurity Rationale**: In normal LAN operations, ARP replies occur only in response to ARP requests. Unsolicited gratuitous ARP replies (MITRE `T1557.002`) are the hallmark of ARP cache poisoning / Adversary-in-the-Middle.
- **Expected Behavior**: Benign traffic has 0 unsolicited ARP replies; ARP spoofing windows show persistent high reply counts.

In [ ]:
fig = px.bar(
    eda_df.groupby('attack_type')['arp_reply_count'].sum().reset_index(),
    x='attack_type',
    y='arp_reply_count',
    title='Total ARP Replies by Traffic Class',
    labels={'arp_reply_count': 'Total ARP Replies (Opcode 2)'},
)
fig.update_layout(template='plotly_dark')
fig.show()

### Feature Group G: Protocol Shares & Inter-Arrival Regularity

- **Cybersecurity Rationale**: High-rate floods distort protocol distributions (e.g., `udp_share` reaching 1.0 during UDP floods).
- **Inter-Arrival Timing**: Scripted flood and scanning tools generate packets with near-deterministic inter-arrival intervals (`inter_arrival_std` $\approx 0$), unlike the variable think time of human web browsing.

In [ ]:
fig = px.scatter(
    eda_df,
    x='inter_arrival_mean',
    y='inter_arrival_std',
    color='attack_type',
    title='Packet Inter-Arrival Mean vs Standard Deviation',
    labels={'inter_arrival_mean': 'Mean Inter-Arrival (s)', 'inter_arrival_std': 'Std Inter-Arrival (s)'},
)
fig.update_layout(template='plotly_dark')
fig.show()

## 3. Bidirectional Flow Analysis

Below we examine flow state distributions across scenarios (`completed`, `reset`, `half_open`).

In [ ]:
flow_summary = flows_df.groupby(['scenario', 'state']).size().unstack(fill_value=0)
print('Flow State Breakdown by Scenario:')
print(flow_summary)

fig = px.bar(
    flows_df,
    x='scenario',
    color='state',
    title='Bidirectional Flow State Classification across Scenarios',
    labels={'scenario': 'Traffic Scenario', 'count': 'Flow Count'},
)
fig.update_layout(template='plotly_dark')
fig.show()

## Summary of Findings

1. **Port Scans (S2)**: Readily identified by `unique_dst_ports` ($>20$) and elevated `short_connection_fraction`.
2. **SYN Floods (S3)**: Characterized by extreme `pps` surges, high `syn_count`, and skewed `syn_without_ack_fraction`.
3. **UDP Floods (S3)**: Characterized by `udp_share` $\approx 1.0$ and high bandwidth `bps`.
4. **SSH Brute Force (S4)**: Readily isolated by `connections_to_same_dst_port_max` on port 22 and `short_connection_fraction`.
5. **ARP Spoofing (S4)**: Directly flagged by `arp_reply_count` and `mac_binding_changes`.
6. **Benign Traffic (S1)**: Demonstrates stable baselines with low variance, making it an ideal reference distribution for training unsupervised Isolation Forests.